In [1]:
# ============================================================
# PHASE 1: SETUP & LIBRARY IMPORTS
# ============================================================
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
import networkx as nx



# Styling
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("="*80)
print("CAUSAL ANALYSIS INITIALIZED")
print("="*80)
print("\nLibraries loaded successfully!\n")

CAUSAL ANALYSIS INITIALIZED

Libraries loaded successfully!



In [7]:
# ============================================================
# PHASE 2: DATA LOADING & INITIAL EXPLORATION
# ============================================================

print("\n" + "="*80)
print("PHASE 2: DATA LOADING")
print("="*80 + "\n")

# Load all datasets
try:
    train1 = pd.read_csv('train1.csv', parse_dates=['date'])
    train2 = pd.read_csv('train2.csv', parse_dates=['date'])
    stores = pd.read_csv('stores.csv')
    holidays = pd.read_csv('holidays_events.csv', parse_dates=['date'])
    transactions = pd.read_csv('transactions.csv', parse_dates=['date'])
    oil = pd.read_csv('oil.csv', parse_dates=['date'])
    
    print("✓ All datasets loaded successfully!")
    print(f"\n📊 Dataset Shapes:")
    print(f"  - train1: {train1.shape}")
    print(f"  - train2: {train2.shape}")
    print(f"  - stores: {stores.shape}")
    print(f"  - holidays: {holidays.shape}")
    print(f"  - transactions: {transactions.shape}")
    print(f"  - oil: {oil.shape}")
    
except Exception as e:
    print(f"❌ Error loading data: {e}")
    print("\nPlease ensure all CSV files are in the current directory:")
    print("  - train1.csv, train2.csv")
    print("  - stores.csv, holidays_events.csv")
    print("  - transactions.csv, oil.csv")


PHASE 2: DATA LOADING

✓ All datasets loaded successfully!

📊 Dataset Shapes:
  - train1: (1500444, 6)
  - train2: (1500444, 6)
  - stores: (54, 5)
  - holidays: (350, 6)
  - transactions: (83488, 3)
  - oil: (1218, 2)


In [10]:
# ============================================================
# PHASE 3: DATA PREPROCESSING & FEATURE ENGINEERING
# ============================================================

print("\n" + "="*80)
print("PHASE 3: DATA PREPROCESSING")
print("="*80 + "\n")

# 3.1 Combine training data
print("Step 3.1: Combining training datasets...")
train = pd.concat([train1, train2], axis=0, ignore_index=True)
print(f"  Combined training data: {train.shape}")

# 3.2 Handle holidays (remove transferred)
print("\nStep 3.2: Processing holidays data...")
holidays_clean = holidays[holidays['transferred'] == False].copy()
print(f"  Original holidays: {len(holidays)}")
print(f"  After removing transferred: {len(holidays_clean)}")

# Create is_holiday binary variable
holidays_clean['is_holiday'] = 1
holidays_agg = holidays_clean.groupby('date').agg({
    'is_holiday': 'max',
    'type': lambda x: ', '.join(x.unique()),
    'locale': lambda x: ', '.join(x.unique())
}).reset_index()

# Rename 'type' column to avoid conflict with stores 'type'
holidays_agg.rename(columns={'type': 'holiday_type'}, inplace=True)

print(f"\n  Holiday types distribution:")
for htype in holidays_clean['type'].value_counts().items():
    print(f"    - {htype[0]}: {htype[1]}")

# 3.3 Oil price interpolation
print("\nStep 3.3: Interpolating oil prices...")
oil_complete = oil.set_index('date')['dcoilwtico'].resample('D').asfreq()
oil_complete = oil_complete.interpolate(method='linear')
oil_df = oil_complete.reset_index()
oil_df.columns = ['date', 'oil_price']
print(f"  Missing values before: {oil['dcoilwtico'].isna().sum()}")
print(f"  Missing values after: {oil_df['oil_price'].isna().sum()}")

# 3.4 Merge all datasets
print("\nStep 3.4: Merging datasets...")
print("  Merging pipeline:")
print("    train → stores → holidays → transactions → oil")

# Merge with stores
data = train.merge(stores, on='store_nbr', how='left')
print(f"  After stores merge: {data.shape}")

# Merge with holidays
data = data.merge(holidays_agg, on='date', how='left')
data['is_holiday'] = data['is_holiday'].fillna(0).astype(int)
print(f"  After holidays merge: {data.shape}")

# Merge with transactions
data = data.merge(transactions, on=['date', 'store_nbr'], how='left')
print(f"  After transactions merge: {data.shape}")

# Merge with oil
data = data.merge(oil_df, on='date', how='left')
print(f"  After oil merge: {data.shape}")

# 3.5 Handle missing values
print("\nStep 3.5: Handling missing values...")
data['transactions'] = data.groupby('store_nbr')['transactions'].transform(
    lambda x: x.fillna(x.median())
)
data['oil_price'] = data['oil_price'].fillna(data['oil_price'].median())

print(f"  Remaining missing values:")
for col in data.columns:
    missing = data[col].isna().sum()
    if missing > 0:
        print(f"    - {col}: {missing}")

# 3.6 Create additional features
print("\nStep 3.6: Feature engineering...")
data['year'] = data['date'].dt.year
data['month'] = data['date'].dt.month
data['dayofweek'] = data['date'].dt.dayofweek
data['quarter'] = data['date'].dt.quarter

# Create holiday type categories
def categorize_holiday_type(row):
    if row['is_holiday'] == 0:
        return 'No Holiday'
    else:
        return row['holiday_type'] if pd.notna(row['holiday_type']) else 'Unknown'

data['holiday_category'] = data.apply(categorize_holiday_type, axis=1)

print("  Created temporal features: year, month, dayofweek, quarter")
print("  Created holiday_category")


PHASE 3: DATA PREPROCESSING

Step 3.1: Combining training datasets...
  Combined training data: (3000888, 6)

Step 3.2: Processing holidays data...
  Original holidays: 350
  After removing transferred: 338

  Holiday types distribution:
    - Holiday: 209
    - Event: 56
    - Additional: 51
    - Transfer: 12
    - Bridge: 5
    - Work Day: 5

Step 3.3: Interpolating oil prices...
  Missing values before: 43
  Missing values after: 1

Step 3.4: Merging datasets...
  Merging pipeline:
    train → stores → holidays → transactions → oil
  After stores merge: (3000888, 10)
  After holidays merge: (3000888, 13)
  After transactions merge: (3000888, 14)
  After oil merge: (3000888, 15)

Step 3.5: Handling missing values...
  Remaining missing values:
    - holiday_type: 2567862
    - locale: 2567862

Step 3.6: Feature engineering...
  Created temporal features: year, month, dayofweek, quarter
  Created holiday_category


In [11]:

# ============================================================
# PHASE 4: EXPLORATORY CAUSAL ANALYSIS
# ============================================================

print("\n" + "="*80)
print("PHASE 4: EXPLORATORY CAUSAL ANALYSIS")
print("="*80 + "\n")

# 4.1 Descriptive statistics
print("Step 4.1: Descriptive Statistics\n")
print("OUTCOME VARIABLE: onpromotion")
print("-" * 40)
print(f"  Overall promotion rate: {data['onpromotion'].mean():.4f}")
print(f"  Std deviation: {data['onpromotion'].std():.4f}")
print(f"\nTREATMENT VARIABLE: is_holiday")
print("-" * 40)
print(f"  Proportion of holiday days: {data['is_holiday'].mean():.4f}")

# 4.2 Unconditional Association (NOT CAUSAL!)
print("\n\nStep 4.2: Unconditional Association (Correlation, NOT Causation)")
print("-" * 40)
promo_holiday = data[data['is_holiday']==1]['onpromotion'].mean()
promo_nonholiday = data[data['is_holiday']==0]['onpromotion'].mean()
naive_diff = promo_holiday - promo_nonholiday

print(f"  Promotion rate on holidays: {promo_holiday:.4f}")
print(f"  Promotion rate on non-holidays: {promo_nonholiday:.4f}")
print(f"  Naive difference: {naive_diff:.4f}")
print(f"\n  ⚠️  WARNING: This is NOT a causal effect!")
print(f"      It includes confounding bias from store characteristics,")
print(f"      seasonal patterns, and other factors.")

# 4.3 Dimension View: Heterogeneity Analysis
print("\n\nStep 4.3: DIMENSION VIEW - Heterogeneity Analysis")
print("="*60)

dimension_results = []

# By Store Type
print("\n📊 By Store Type:")
for stype in data['type'].unique():
    subset = data[data['type'] == stype]
    promo_hol = subset[subset['is_holiday']==1]['onpromotion'].mean()
    promo_nohol = subset[subset['is_holiday']==0]['onpromotion'].mean()
    diff = promo_hol - promo_nohol
    n_obs = len(subset)
    
    dimension_results.append({
        'Dimension': 'Store Type',
        'Category': stype,
        'Promo_Holiday': promo_hol,
        'Promo_NonHoliday': promo_nohol,
        'Difference': diff,
        'N_Observations': n_obs
    })
    
    print(f"  {stype:15s} | Holiday: {promo_hol:.4f} | Non-Holiday: {promo_nohol:.4f} | Diff: {diff:+.4f}")

# By Cluster
print("\n📊 By Store Cluster:")
for cluster in sorted(data['cluster'].unique()):
    subset = data[data['cluster'] == cluster]
    promo_hol = subset[subset['is_holiday']==1]['onpromotion'].mean()
    promo_nohol = subset[subset['is_holiday']==0]['onpromotion'].mean()
    diff = promo_hol - promo_nohol
    n_obs = len(subset)
    
    dimension_results.append({
        'Dimension': 'Cluster',
        'Category': f'Cluster {cluster}',
        'Promo_Holiday': promo_hol,
        'Promo_NonHoliday': promo_nohol,
        'Difference': diff,
        'N_Observations': n_obs
    })
    
    print(f"  Cluster {cluster:2d}     | Holiday: {promo_hol:.4f} | Non-Holiday: {promo_nohol:.4f} | Diff: {diff:+.4f}")

# By Holiday Type
print("\n📊 By Holiday Type:")
for htype in data['holiday_category'].unique():
    if htype == 'No Holiday':
        continue
    subset = data[data['holiday_category'] == htype]
    promo_rate = subset['onpromotion'].mean()
    n_obs = len(subset)
    
    dimension_results.append({
        'Dimension': 'Holiday Type',
        'Category': htype,
        'Promo_Holiday': promo_rate,
        'Promo_NonHoliday': np.nan,
        'Difference': np.nan,
        'N_Observations': n_obs
    })
    
    print(f"  {htype:20s} | Promo Rate: {promo_rate:.4f} | N: {n_obs}")

# Save dimension results
dimension_df = pd.DataFrame(dimension_results)
print(f"\n✓ Dimension analysis complete. Total dimensions analyzed: {len(dimension_df)}")

# 4.4 Correlation Matrix (NOT CAUSAL)
print("\n\nStep 4.4: Correlation Matrix (Associational, NOT Causal)")
print("="*60)

numeric_cols = ['onpromotion', 'is_holiday', 'transactions', 'oil_price', 
                'cluster', 'month', 'dayofweek']
corr_matrix = data[numeric_cols].corr()

print("\nKey correlations with onpromotion:")
corr_with_outcome = corr_matrix['onpromotion'].sort_values(ascending=False)
for idx, val in corr_with_outcome.items():
    if idx != 'onpromotion':
        print(f"  {idx:20s}: {val:+.4f}")



PHASE 4: EXPLORATORY CAUSAL ANALYSIS

Step 4.1: Descriptive Statistics

OUTCOME VARIABLE: onpromotion
----------------------------------------
  Overall promotion rate: 2.6028
  Std deviation: 12.2189

TREATMENT VARIABLE: is_holiday
----------------------------------------
  Proportion of holiday days: 0.1443


Step 4.2: Unconditional Association (Correlation, NOT Causation)
----------------------------------------
  Promotion rate on holidays: 3.0368
  Promotion rate on non-holidays: 2.5296
  Naive difference: 0.5073

  ⚠️  WARNING: This is NOT a causal effect!
      It includes confounding bias from store characteristics,
      seasonal patterns, and other factors.


Step 4.3: DIMENSION VIEW - Heterogeneity Analysis

📊 By Store Type:
  D               | Holiday: 3.3518 | Non-Holiday: 2.5888 | Diff: +0.7630
  C               | Holiday: 2.3830 | Non-Holiday: 2.0407 | Diff: +0.3423
  B               | Holiday: 3.0623 | Non-Holiday: 2.8005 | Diff: +0.2618
  E               | Holiday: 2.

In [12]:

# ============================================================
# PHASE 5: CAUSAL IDENTIFICATION
# ============================================================

print("\n\n" + "="*80)
print("PHASE 5: CAUSAL IDENTIFICATION")
print("="*80 + "\n")

# 5.1 Propensity Score Estimation
print("Step 5.1: Propensity Score Estimation")
print("-" * 40)

# Select confounders (variables that affect both treatment and outcome)
confounders = ['cluster', 'oil_price', 'month', 'dayofweek', 'transactions']

# Encode categorical variables
data_ps = data.copy()
data_ps['type_encoded'] = pd.Categorical(data_ps['type']).codes

# Prepare features for propensity score
X_ps = data_ps[confounders + ['type_encoded']].copy()
X_ps = X_ps.fillna(X_ps.median())  # Handle any remaining NaN

# Fit logistic regression for propensity score
ps_model = LogisticRegression(max_iter=1000, random_state=42)
ps_model.fit(X_ps, data_ps['is_holiday'])

# Predict propensity scores
data_ps['propensity_score'] = ps_model.predict_proba(X_ps)[:, 1]

print("  ✓ Propensity score model fitted")
print(f"  Confounders used: {', '.join(confounders + ['type_encoded'])}")
print(f"\n  Propensity Score Summary:")
print(f"    Mean: {data_ps['propensity_score'].mean():.4f}")
print(f"    Std:  {data_ps['propensity_score'].std():.4f}")
print(f"    Min:  {data_ps['propensity_score'].min():.4f}")
print(f"    Max:  {data_ps['propensity_score'].max():.4f}")

# 5.2 Overlap/Positivity Check
print("\n\nStep 5.2: Overlap Assessment (Positivity Assumption)")
print("-" * 40)

ps_treated = data_ps[data_ps['is_holiday']==1]['propensity_score']
ps_control = data_ps[data_ps['is_holiday']==0]['propensity_score']

print(f"  Treated group (holidays):")
print(f"    N = {len(ps_treated)}")
print(f"    PS range: [{ps_treated.min():.4f}, {ps_treated.max():.4f}]")
print(f"\n  Control group (non-holidays):")
print(f"    N = {len(ps_control)}")
print(f"    PS range: [{ps_control.min():.4f}, {ps_control.max():.4f}]")

# Check common support
common_support_min = max(ps_treated.min(), ps_control.min())
common_support_max = min(ps_treated.max(), ps_control.max())
print(f"\n  Common support region: [{common_support_min:.4f}, {common_support_max:.4f}]")

# Trim data to common support
data_trimmed = data_ps[
    (data_ps['propensity_score'] >= common_support_min) &
    (data_ps['propensity_score'] <= common_support_max)
].copy()
print(f"  Observations in common support: {len(data_trimmed)} ({len(data_trimmed)/len(data_ps)*100:.1f}%)")

# 5.3 Balance Checking
print("\n\nStep 5.3: Covariate Balance Check")
print("-" * 40)

balance_results = []

for var in confounders + ['type_encoded']:
    # Before matching
    treated_mean = data_ps[data_ps['is_holiday']==1][var].mean()
    control_mean = data_ps[data_ps['is_holiday']==0][var].mean()
    pooled_std = np.sqrt(
        (data_ps[data_ps['is_holiday']==1][var].var() + 
         data_ps[data_ps['is_holiday']==0][var].var()) / 2
    )
    smd_before = (treated_mean - control_mean) / pooled_std if pooled_std > 0 else 0
    
    balance_results.append({
        'Variable': var,
        'SMD_Before': smd_before,
        'Treated_Mean': treated_mean,
        'Control_Mean': control_mean,
        'Balanced': 'Yes' if abs(smd_before) < 0.1 else 'No'
    })
    
    status = '✓' if abs(smd_before) < 0.1 else '✗'
    print(f"  {status} {var:20s} | SMD: {smd_before:+.4f} | Balanced: {abs(smd_before) < 0.1}")

balance_df = pd.DataFrame(balance_results)
n_balanced = (balance_df['SMD_Before'].abs() < 0.1).sum()
print(f"\n  Balanced variables (|SMD| < 0.1): {n_balanced}/{len(balance_df)}")




PHASE 5: CAUSAL IDENTIFICATION

Step 5.1: Propensity Score Estimation
----------------------------------------
  ✓ Propensity score model fitted
  Confounders used: cluster, oil_price, month, dayofweek, transactions, type_encoded

  Propensity Score Summary:
    Mean: 0.1443
    Std:  0.0477
    Min:  0.0676
    Max:  0.4464


Step 5.2: Overlap Assessment (Positivity Assumption)
----------------------------------------
  Treated group (holidays):
    N = 433026
    PS range: [0.0688, 0.4464]

  Control group (non-holidays):
    N = 2567862
    PS range: [0.0676, 0.3956]

  Common support region: [0.0688, 0.3956]
  Observations in common support: 2998875 (99.9%)


Step 5.3: Covariate Balance Check
----------------------------------------
  ✓ cluster              | SMD: +0.0000 | Balanced: True
  ✓ oil_price            | SMD: -0.0779 | Balanced: True
  ✗ month                | SMD: +0.3502 | Balanced: False
  ✓ dayofweek            | SMD: +0.0017 | Balanced: True
  ✗ transactions      

In [13]:

# ============================================================
# PHASE 6: CAUSAL ESTIMATION
# ============================================================

print("\n\n" + "="*80)
print("PHASE 6: CAUSAL ESTIMATION")
print("="*80 + "\n")

# 6.1 Linear Probability Model (LPM)
print("Step 6.1: Linear Probability Model (OLS Regression)")
print("-" * 60)

from sklearn.linear_model import LinearRegression
from scipy.stats import t as t_dist

# Prepare data for regression
data_reg = data_ps.copy()
data_reg['type_encoded'] = pd.Categorical(data_reg['type']).codes

# Model 1: Naive (no controls)
X_naive = data_reg[['is_holiday']].values
y = data_reg['onpromotion'].values

model_naive = LinearRegression()
model_naive.fit(X_naive, y)
ate_naive = model_naive.coef_[0]

print("\n📊 MODEL 1: Naive Regression (No Controls)")
print(f"  onpromotion ~ is_holiday")
print(f"  Coefficient (β): {ate_naive:.6f}")
print(f"  ⚠️  This is BIASED due to confounding!\n")

# Model 2: Adjusted (with confounders)
control_vars = ['is_holiday', 'cluster', 'oil_price', 'month', 
                'dayofweek', 'transactions', 'type_encoded']
X_adjusted = data_reg[control_vars].fillna(data_reg[control_vars].median()).values

model_adjusted = LinearRegression()
model_adjusted.fit(X_adjusted, y)
ate_adjusted = model_adjusted.coef_[0]

# Calculate standard errors
n = len(y)
k = X_adjusted.shape[1]
y_pred = model_adjusted.predict(X_adjusted)
residuals = y - y_pred
mse = np.sum(residuals**2) / (n - k)
var_coef = mse * np.linalg.inv(X_adjusted.T @ X_adjusted).diagonal()
se_adjusted = np.sqrt(var_coef)

# 95% CI
t_critical = t_dist.ppf(0.975, n - k)
ci_lower = ate_adjusted - t_critical * se_adjusted[0]
ci_upper = ate_adjusted + t_critical * se_adjusted[0]

# p-value
t_stat = ate_adjusted / se_adjusted[0]
p_value = 2 * (1 - t_dist.cdf(abs(t_stat), n - k))

print("\n📊 MODEL 2: Adjusted Regression (With Confounders)")
print(f"  onpromotion ~ is_holiday + cluster + oil_price + month + dayofweek + transactions + store_type")
print(f"\n  CAUSAL ESTIMATE (ATE):")
print(f"    Coefficient (β):  {ate_adjusted:.6f}")
print(f"    Standard Error:   {se_adjusted[0]:.6f}")
print(f"    95% CI:          [{ci_lower:.6f}, {ci_upper:.6f}]")
print(f"    t-statistic:      {t_stat:.4f}")
print(f"    p-value:          {p_value:.4e}")
print(f"\n  INTERPRETATION:")
if p_value < 0.05:
    print(f"    ✓ Holiday CAUSALLY increases promotion probability by {ate_adjusted*100:.2f} percentage points")
    print(f"      (p < 0.05, statistically significant)")
else:
    print(f"    ✗ No significant causal effect detected (p ≥ 0.05)")

# Display all coefficients
print(f"\n  All Coefficients:")
for i, var in enumerate(control_vars):
    print(f"    {var:20s}: {model_adjusted.coef_[i]:+.6f} (SE: {se_adjusted[i]:.6f})")

# 6.2 Inverse Probability Weighting (IPW)
print("\n\nStep 6.2: Inverse Probability Weighting (IPW) Estimator")
print("-" * 60)

# Calculate IPW weights
data_ps['ipw_weight'] = np.where(
    data_ps['is_holiday'] == 1,
    1 / data_ps['propensity_score'],
    1 / (1 - data_ps['propensity_score'])
)

# Trim extreme weights
weight_99 = data_ps['ipw_weight'].quantile(0.99)
data_ps['ipw_weight_trimmed'] = data_ps['ipw_weight'].clip(upper=weight_99)

# Calculate IPW estimate
ate_ipw = (
    (data_ps[data_ps['is_holiday']==1]['onpromotion'] * 
     data_ps[data_ps['is_holiday']==1]['ipw_weight_trimmed']).mean() -
    (data_ps[data_ps['is_holiday']==0]['onpromotion'] * 
     data_ps[data_ps['is_holiday']==0]['ipw_weight_trimmed']).mean()
)

print(f"  IPW Estimate (ATE): {ate_ipw:.6f}")
print(f"  Weight summary:")
print(f"    Mean: {data_ps['ipw_weight_trimmed'].mean():.2f}")
print(f"    Max:  {data_ps['ipw_weight_trimmed'].max():.2f}")

# 6.3 Difference-in-Differences (DiD)
print("\n\nStep 6.3: Difference-in-Differences (Quasi-Experimental)")
print("-" * 60)

# Create pre/post periods around holidays
data_did = data_ps.copy()
data_did = data_did.sort_values(['store_nbr', 'date'])

# Define treatment as major holidays (type='Holiday')
major_holidays = holidays_clean[holidays_clean['type']=='Holiday']['date'].unique()

did_results = []

for holiday in major_holidays[:5]:  # Analyze first 5 major holidays
    # Pre-period: 7 days before
    # Post-period: 7 days after
    pre_start = holiday - pd.Timedelta(days=7)
    post_end = holiday + pd.Timedelta(days=7)
    
    subset = data_did[
        (data_did['date'] >= pre_start) & 
        (data_did['date'] <= post_end)
    ].copy()
    
    subset['post'] = (subset['date'] >= holiday).astype(int)
    subset['treated_store'] = (subset['is_holiday'] == 1).astype(int)
    
    # DiD regression: Y = β0 + β1*Post + β2*Treated + β3*(Post×Treated)
    if len(subset) > 0:
        X_did = subset[['post', 'treated_store']].copy()
        X_did['post_x_treated'] = X_did['post'] * X_did['treated_store']
        
        model_did = LinearRegression()
        model_did.fit(X_did, subset['onpromotion'])
        
        did_estimate = model_did.coef_[2]  # Interaction coefficient
        
        did_results.append({
            'holiday': holiday,
            'did_estimate': did_estimate
        })

if did_results:
    avg_did = np.mean([r['did_estimate'] for r in did_results])
    print(f"  Average DiD Estimate: {avg_did:.6f}")
    print(f"  (Based on {len(did_results)} major holidays)")



PHASE 6: CAUSAL ESTIMATION

Step 6.1: Linear Probability Model (OLS Regression)
------------------------------------------------------------

📊 MODEL 1: Naive Regression (No Controls)
  onpromotion ~ is_holiday
  Coefficient (β): 0.507258
  ⚠️  This is BIASED due to confounding!


📊 MODEL 2: Adjusted Regression (With Confounders)
  onpromotion ~ is_holiday + cluster + oil_price + month + dayofweek + transactions + store_type

  CAUSAL ESTIMATE (ATE):
    Coefficient (β):  0.195991
    Standard Error:   0.019989
    95% CI:          [0.156814, 0.235168]
    t-statistic:      9.8051
    p-value:          0.0000e+00

  INTERPRETATION:
    ✓ Holiday CAUSALLY increases promotion probability by 19.60 percentage points
      (p < 0.05, statistically significant)

  All Coefficients:
    is_holiday          : +0.195991 (SE: 0.019989)
    cluster             : +0.002899 (SE: 0.001431)
    oil_price           : -0.073410 (SE: 0.000236)
    month               : +0.092459 (SE: 0.001972)
    day

In [14]:


# ============================================================
# PHASE 7: MEDIATION ANALYSIS
# ============================================================

print("\n\n" + "="*80)
print("PHASE 7: MEDIATION ANALYSIS")
print("="*80 + "\n")

print("Mediation Path: is_holiday → transactions → onpromotion")
print("-" * 60)

# Step 1: Total effect (c)
X_total = data_reg[['is_holiday']].values
model_total = LinearRegression()
model_total.fit(X_total, y)
total_effect = model_total.coef_[0]

print(f"\n  Total Effect (c):       {total_effect:.6f}")

# Step 2: Effect on mediator (a)
X_mediator = data_reg[['is_holiday']].values
y_mediator = data_reg['transactions'].fillna(data_reg['transactions'].median()).values
model_mediator = LinearRegression()
model_mediator.fit(X_mediator, y_mediator)
effect_a = model_mediator.coef_[0]

print(f"  Effect on Mediator (a): {effect_a:.6f}")

# Step 3: Direct effect (c')
X_direct = data_reg[['is_holiday', 'transactions']].fillna(data_reg[['is_holiday', 'transactions']].median()).values
model_direct = LinearRegression()
model_direct.fit(X_direct, y)
direct_effect = model_direct.coef_[0]
effect_b = model_direct.coef_[1]

print(f"  Mediator → Outcome (b): {effect_b:.6f}")
print(f"  Direct Effect (c'):     {direct_effect:.6f}")

# Step 4: Indirect effect
indirect_effect = effect_a * effect_b
mediation_proportion = indirect_effect / total_effect if total_effect != 0 else 0

print(f"\n  Indirect Effect (a×b):  {indirect_effect:.6f}")
print(f"  Mediation Proportion:   {mediation_proportion*100:.2f}%")

print(f"\n  INTERPRETATION:")
print(f"    {mediation_proportion*100:.1f}% of the holiday effect on promotions")
print(f"    is mediated through customer transactions.")



PHASE 7: MEDIATION ANALYSIS

Mediation Path: is_holiday → transactions → onpromotion
------------------------------------------------------------

  Total Effect (c):       0.507258
  Effect on Mediator (a): 122.079093
  Mediator → Outcome (b): 0.000406
  Direct Effect (c'):     0.457685

  Indirect Effect (a×b):  0.049573
  Mediation Proportion:   9.77%

  INTERPRETATION:
    9.8% of the holiday effect on promotions
    is mediated through customer transactions.


In [15]:


# ============================================================
# PHASE 8: COUNTERFACTUAL ANALYSIS
# ============================================================

print("\n\n" + "="*80)
print("PHASE 8: COUNTERFACTUAL ANALYSIS")
print("="*80 + "\n")

# 8.1 Individual Treatment Effects (ITE)
print("Step 8.1: Individual Treatment Effect Estimation")
print("-" * 60)

# Predict potential outcomes
X_full = data_reg[control_vars[1:]].fillna(data_reg[control_vars[1:]].median()).values

# Y(1) - potential outcome if treated
X_treated = np.column_stack([np.ones(len(X_full)), X_full])
y1_pred = model_adjusted.predict(X_treated)

# Y(0) - potential outcome if control
X_control = np.column_stack([np.zeros(len(X_full)), X_full])
y0_pred = model_adjusted.predict(X_control)

# ITE
data_reg['ite'] = y1_pred - y0_pred

print(f"  Individual Treatment Effects:")
print(f"    Mean ITE:  {data_reg['ite'].mean():.6f}")
print(f"    Std ITE:   {data_reg['ite'].std():.6f}")
print(f"    Min ITE:   {data_reg['ite'].min():.6f}")
print(f"    Max ITE:   {data_reg['ite'].max():.6f}")

# Heterogeneous effects
print(f"\n  ITE by Store Type:")
for stype in data_reg['type'].unique():
    ite_mean = data_reg[data_reg['type']==stype]['ite'].mean()
    print(f"    {stype:15s}: {ite_mean:.6f}")

# 8.2 Policy Simulation
print("\n\nStep 8.2: Policy Simulation")
print("-" * 60)

# Scenario 1: Current state
current_promo_rate = data_reg['onpromotion'].mean()

# Scenario 2: If all days were holidays
all_holiday_promo = y1_pred.mean()

# Scenario 3: If no holidays
no_holiday_promo = y0_pred.mean()

print(f"  Policy Scenarios:")
print(f"    Current state:        {current_promo_rate:.4f}")
print(f"    All days = holidays:  {all_holiday_promo:.4f} (Δ = {(all_holiday_promo-current_promo_rate):.4f})")
print(f"    No holidays:          {no_holiday_promo:.4f} (Δ = {(no_holiday_promo-current_promo_rate):.4f})")




PHASE 8: COUNTERFACTUAL ANALYSIS

Step 8.1: Individual Treatment Effect Estimation
------------------------------------------------------------
  Individual Treatment Effects:
    Mean ITE:  0.195991
    Std ITE:   0.000000
    Min ITE:   0.195991
    Max ITE:   0.195991

  ITE by Store Type:
    D              : 0.195991
    C              : 0.195991
    B              : 0.195991
    E              : 0.195991
    A              : 0.195991


Step 8.2: Policy Simulation
------------------------------------------------------------
  Policy Scenarios:
    Current state:        2.6028
    All days = holidays:  2.7705 (Δ = 0.1677)
    No holidays:          2.5745 (Δ = -0.0283)


In [16]:
# ============================================================
# PHASE 9: SUMMARY TABLES FOR UI
# ============================================================

print("\n\n" + "="*80)
print("PHASE 9: GENERATING OUTPUT TABLES FOR UI")
print("="*80 + "\n")

# Table 1: Dimension View
print("✓ Table 1: DIMENSION VIEW")
print(dimension_df.to_string(index=False))

# Table 2: Causal Estimates
causal_estimates = pd.DataFrame({
    'Method': ['Naive OLS', 'Adjusted OLS', 'IPW', 'DiD (Avg)'],
    'Estimate': [ate_naive, ate_adjusted, ate_ipw, avg_did if did_results else np.nan],
    'Interpretation': [
        'Biased (no controls)',
        'Causal (backdoor adjustment)',
        'Causal (reweighting)',
        'Causal (quasi-experimental)'
    ]
})

print("\n✓ Table 2: CAUSAL ESTIMATES")
print(causal_estimates.to_string(index=False))

# Table 3: Balance Table
print("\n✓ Table 3: COVARIATE BALANCE")
print(balance_df.to_string(index=False))

# Table 4: Regression Output
regression_output = pd.DataFrame({
    'Variable': control_vars,
    'Coefficient': model_adjusted.coef_,
    'Std_Error': se_adjusted,
    't_statistic': model_adjusted.coef_ / se_adjusted,
    'p_value': [2 * (1 - t_dist.cdf(abs(model_adjusted.coef_[i] / se_adjusted[i]), n - k)) 
                for i in range(len(control_vars))]
})
regression_output['Significance'] = regression_output['p_value'].apply(
    lambda p: '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.1 else ''))
)

print("\n✓ Table 4: REGRESSION OUTPUT (LPM)")
print(regression_output.to_string(index=False))

# Table 5: Mediation Analysis
mediation_table = pd.DataFrame({
    'Path': ['Total Effect (c)', 'Direct Effect (c\')', 'Indirect Effect (a×b)', 'Effect on Mediator (a)', 'Mediator to Outcome (b)'],
    'Coefficient': [total_effect, direct_effect, indirect_effect, effect_a, effect_b],
    'Percentage': [100.0, (direct_effect/total_effect)*100 if total_effect != 0 else 0, 
                   mediation_proportion*100, np.nan, np.nan]
})

print("\n✓ Table 5: MEDIATION ANALYSIS")
print(mediation_table.to_string(index=False))

# Table 6: Heterogeneous Treatment Effects
hte_table = []
for stype in data_reg['type'].unique():
    subset = data_reg[data_reg['type']==stype]
    hte_table.append({
        'Dimension': 'Store Type',
        'Category': stype,
        'Mean_ITE': subset['ite'].mean(),
        'Std_ITE': subset['ite'].std(),
        'N': len(subset)
    })

for cluster in sorted(data_reg['cluster'].unique())[:10]:  # Top 10 clusters
    subset = data_reg[data_reg['cluster']==cluster]
    hte_table.append({
        'Dimension': 'Cluster',
        'Category': f'Cluster {cluster}',
        'Mean_ITE': subset['ite'].mean(),
        'Std_ITE': subset['ite'].std(),
        'N': len(subset)
    })

hte_df = pd.DataFrame(hte_table)
print("\n✓ Table 6: HETEROGENEOUS TREATMENT EFFECTS")
print(hte_df.to_string(index=False))




PHASE 9: GENERATING OUTPUT TABLES FOR UI

✓ Table 1: DIMENSION VIEW
   Dimension             Category  Promo_Holiday  Promo_NonHoliday  Difference  N_Observations
  Store Type                    D       3.351769          2.588750    0.763019         1000296
  Store Type                    C       2.382974          2.040700    0.342274          833580
  Store Type                    B       3.062305          2.800507    0.261798          444576
  Store Type                    E       2.898616          2.674505    0.224110          222288
  Store Type                    A       3.535506          2.920764    0.614742          500148
     Cluster            Cluster 1       3.045226          2.451412    0.593814          166716
     Cluster            Cluster 2       2.871181          2.415421    0.455760          111144
     Cluster            Cluster 3       2.539825          2.083846    0.455979          389004
     Cluster            Cluster 4       3.098973          2.547123    0.551

In [17]:

# ============================================================
# PHASE 10: VISUALIZATION OUTPUTS
# ============================================================

print("\n\n" + "="*80)
print("PHASE 10: GENERATING VISUALIZATIONS")
print("="*80 + "\n")

# Figure 1: Propensity Score Overlap
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: PS Distribution
axes[0, 0].hist(ps_control, bins=50, alpha=0.6, label='Control', density=True, color='blue')
axes[0, 0].hist(ps_treated, bins=50, alpha=0.6, label='Treated', density=True, color='red')
axes[0, 0].set_xlabel('Propensity Score')
axes[0, 0].set_ylabel('Density')
axes[0, 0].set_title('Propensity Score Overlap (Positivity Check)')
axes[0, 0].legend()
axes[0, 0].axvline(common_support_min, color='green', linestyle='--', label='Common Support')
axes[0, 0].axvline(common_support_max, color='green', linestyle='--')

# Plot 2: Balance Plot
balance_df_sorted = balance_df.sort_values('SMD_Before', key=abs, ascending=False)
colors = ['green' if abs(x) < 0.1 else 'red' for x in balance_df_sorted['SMD_Before']]
axes[0, 1].barh(balance_df_sorted['Variable'], balance_df_sorted['SMD_Before'], color=colors, alpha=0.7)
axes[0, 1].axvline(-0.1, color='black', linestyle='--', linewidth=0.8)
axes[0, 1].axvline(0.1, color='black', linestyle='--', linewidth=0.8)
axes[0, 1].axvline(0, color='black', linewidth=1.5)
axes[0, 1].set_xlabel('Standardized Mean Difference')
axes[0, 1].set_title('Covariate Balance (|SMD| < 0.1 = Balanced)')

# Plot 3: Treatment Effect by Store Type
dim_storetype = dimension_df[dimension_df['Dimension']=='Store Type'].copy()
axes[1, 0].bar(dim_storetype['Category'], dim_storetype['Difference'], 
               color=['green' if x > 0 else 'red' for x in dim_storetype['Difference']], alpha=0.7)
axes[1, 0].set_ylabel('Promotion Rate Difference')
axes[1, 0].set_title('Treatment Effect by Store Type')
axes[1, 0].axhline(0, color='black', linewidth=1)
axes[1, 0].tick_params(axis='x', rotation=45)

# Plot 4: ITE Distribution
axes[1, 1].hist(data_reg['ite'], bins=50, color='purple', alpha=0.7, edgecolor='black')
axes[1, 1].axvline(data_reg['ite'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean ITE = {data_reg['ite'].mean():.4f}")
axes[1, 1].set_xlabel('Individual Treatment Effect')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Distribution of Individual Treatment Effects')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('causal_analysis_plots_1.png', dpi=300, bbox_inches='tight')
print("✓ Figure 1 saved: causal_analysis_plots_1.png")
plt.close()

# Figure 2: Causal Graph & Results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Time Series of Promotion Rate
daily_stats = data_reg.groupby('date').agg({
    'onpromotion': 'mean',
    'is_holiday': 'max'
}).reset_index()

axes[0, 0].plot(daily_stats['date'], daily_stats['onpromotion'], linewidth=0.5, alpha=0.7, color='blue')
holiday_dates = daily_stats[daily_stats['is_holiday']==1]['date']
for hdate in holiday_dates:
    axes[0, 0].axvline(hdate, color='red', alpha=0.2, linewidth=0.5)
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Promotion Rate')
axes[0, 0].set_title('Promotion Rate Over Time (Red = Holidays)')

# Plot 2: Effect Size Comparison
methods = causal_estimates['Method'].tolist()
estimates = causal_estimates['Estimate'].tolist()
colors_methods = ['gray', 'green', 'blue', 'orange']
axes[0, 1].barh(methods, estimates, color=colors_methods, alpha=0.7)
axes[0, 1].axvline(0, color='black', linewidth=1.5)
axes[0, 1].set_xlabel('Average Treatment Effect')
axes[0, 1].set_title('Causal Estimates Comparison')

# Plot 3: Mediation Visualization
mediation_labels = ['Total\nEffect', 'Direct\nEffect', 'Indirect\nEffect']
mediation_values = [total_effect, direct_effect, indirect_effect]
mediation_colors = ['purple', 'blue', 'orange']
axes[1, 0].bar(mediation_labels, mediation_values, color=mediation_colors, alpha=0.7)
axes[1, 0].set_ylabel('Effect Size')
axes[1, 0].set_title('Mediation Analysis: Direct vs Indirect Effects')
axes[1, 0].axhline(0, color='black', linewidth=1)

# Plot 4: Policy Simulation
scenarios = ['Current\nState', 'All Days\nHolidays', 'No\nHolidays']
scenario_values = [current_promo_rate, all_holiday_promo, no_holiday_promo]
scenario_colors = ['gray', 'green', 'red']
axes[1, 1].bar(scenarios, scenario_values, color=scenario_colors, alpha=0.7)
axes[1, 1].set_ylabel('Promotion Rate')
axes[1, 1].set_title('Policy Simulation: Counterfactual Scenarios')
axes[1, 1].set_ylim([min(scenario_values)*0.95, max(scenario_values)*1.05])

for i, v in enumerate(scenario_values):
    axes[1, 1].text(i, v + 0.001, f'{v:.4f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('causal_analysis_plots_2.png', dpi=300, bbox_inches='tight')
print("✓ Figure 2 saved: causal_analysis_plots_2.png")
plt.close()

# Figure 3: Detailed Dimension Analysis
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

# Plot 1: Heatmap of promotion rates
ax1 = fig.add_subplot(gs[0, :])
pivot_data = data_reg.groupby(['type', 'is_holiday'])['onpromotion'].mean().unstack()
sns.heatmap(pivot_data, annot=True, fmt='.4f', cmap='RdYlGn', ax=ax1, cbar_kws={'label': 'Promotion Rate'})
ax1.set_title('Promotion Rate by Store Type and Holiday Status')
ax1.set_xlabel('Is Holiday')
ax1.set_ylabel('Store Type')

# Plot 2: Cluster Effects
ax2 = fig.add_subplot(gs[1, 0])
cluster_effects = data_reg.groupby('cluster')['ite'].mean().sort_values()
cluster_effects.plot(kind='barh', ax=ax2, color='teal', alpha=0.7)
ax2.set_xlabel('Mean Individual Treatment Effect')
ax2.set_title('Heterogeneous Effects by Store Cluster')
ax2.axvline(0, color='black', linewidth=1)

# Plot 3: Monthly Pattern
ax3 = fig.add_subplot(gs[1, 1])
monthly_holiday = data_reg[data_reg['is_holiday']==1].groupby('month')['onpromotion'].mean()
monthly_nonholiday = data_reg[data_reg['is_holiday']==0].groupby('month')['onpromotion'].mean()
months = range(1, 13)
ax3.plot(months, [monthly_holiday.get(m, 0) for m in months], marker='o', label='Holiday', linewidth=2)
ax3.plot(months, [monthly_nonholiday.get(m, 0) for m in months], marker='s', label='Non-Holiday', linewidth=2)
ax3.set_xlabel('Month')
ax3.set_ylabel('Promotion Rate')
ax3.set_title('Seasonal Pattern of Promotion')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Oil Price Confounder
ax4 = fig.add_subplot(gs[2, 0])
oil_bins = pd.qcut(data_reg['oil_price'], q=5, duplicates='drop')
oil_effect = data_reg.groupby([oil_bins, 'is_holiday'])['onpromotion'].mean().unstack()
oil_effect.plot(kind='bar', ax=ax4, color=['blue', 'red'], alpha=0.7)
ax4.set_xlabel('Oil Price Quintile')
ax4.set_ylabel('Promotion Rate')
ax4.set_title('Promotion Rate by Oil Price (Confounder)')
ax4.legend(['Non-Holiday', 'Holiday'])
ax4.tick_params(axis='x', rotation=45)

# Plot 5: Transaction Volume Mediator
ax5 = fig.add_subplot(gs[2, 1])
trans_bins = pd.qcut(data_reg['transactions'].fillna(0), q=5, duplicates='drop')
trans_effect = data_reg.groupby([trans_bins, 'is_holiday'])['onpromotion'].mean().unstack()
trans_effect.plot(kind='bar', ax=ax5, color=['blue', 'red'], alpha=0.7)
ax5.set_xlabel('Transaction Volume Quintile')
ax5.set_ylabel('Promotion Rate')
ax5.set_title('Promotion Rate by Transactions (Mediator)')
ax5.legend(['Non-Holiday', 'Holiday'])
ax5.tick_params(axis='x', rotation=45)

plt.savefig('causal_analysis_plots_3.png', dpi=300, bbox_inches='tight')
print("✓ Figure 3 saved: causal_analysis_plots_3.png")
plt.close()



PHASE 10: GENERATING VISUALIZATIONS

✓ Figure 1 saved: causal_analysis_plots_1.png
✓ Figure 2 saved: causal_analysis_plots_2.png
✓ Figure 3 saved: causal_analysis_plots_3.png


In [ ]:

# ============================================================
# PHASE 11: EXPORT DATA FOR UI
# ============================================================

print("\n\n" + "="*80)
print("PHASE 11: EXPORTING DATA FOR UI COMPONENTS")
print("="*80 + "\n")

# Export dimension view data
dimension_df.to_csv('output_dimension_view.csv', index=False)
print("✓ Exported: output_dimension_view.csv")

# Export causal estimates
causal_estimates.to_csv('output_causal_estimates.csv', index=False)
print("✓ Exported: output_causal_estimates.csv")

# Export balance table
balance_df.to_csv('output_balance_table.csv', index=False)
print("✓ Exported: output_balance_table.csv")

# Export regression output
regression_output.to_csv('output_regression_table.csv', index=False)
print("✓ Exported: output_regression_table.csv")

# Export mediation analysis
mediation_table.to_csv('output_mediation_analysis.csv', index=False)
print("✓ Exported: output_mediation_analysis.csv")

# Export heterogeneous effects
hte_df.to_csv('output_heterogeneous_effects.csv', index=False)
print("✓ Exported: output_heterogeneous_effects.csv")

# Export processed data with ITE for graph view
graph_data = data_reg[['date', 'store_nbr', 'is_holiday', 'onpromotion', 
                        'type', 'cluster', 'oil_price', 'transactions', 
                        'propensity_score', 'ite']].copy()
graph_data.to_csv('output_graph_data.csv', index=False)
print("✓ Exported: output_graph_data.csv")

# ============================================================
# PHASE 12: CAUSAL GRAPH SPECIFICATION
# ============================================================

print("\n\n" + "="*80)
print("PHASE 12: CAUSAL GRAPH SPECIFICATION (DAG)")
print("="*80 + "\n")

# Define DAG structure
causal_graph = {
    'nodes': [
        {'id': 'is_holiday', 'label': 'Holiday\n(Treatment)', 'type': 'treatment'},
        {'id': 'onpromotion', 'label': 'Promotion\n(Outcome)', 'type': 'outcome'},
        {'id': 'store_type', 'label': 'Store Type\n(Confounder)', 'type': 'confounder'},
        {'id': 'cluster', 'label': 'Cluster\n(Confounder)', 'type': 'confounder'},
        {'id': 'oil_price', 'label': 'Oil Price\n(Confounder)', 'type': 'confounder'},
        {'id': 'month', 'label': 'Month\n(Confounder)', 'type': 'confounder'},
        {'id': 'transactions', 'label': 'Transactions\n(Mediator)', 'type': 'mediator'}
    ],
    'edges': [
        {'from': 'is_holiday', 'to': 'onpromotion', 'type': 'direct', 'weight': direct_effect},
        {'from': 'is_holiday', 'to': 'transactions', 'type': 'mediator', 'weight': effect_a},
        {'from': 'transactions', 'to': 'onpromotion', 'type': 'mediator', 'weight': effect_b},
        {'from': 'store_type', 'to': 'is_holiday', 'type': 'confounder', 'weight': None},
        {'from': 'store_type', 'to': 'onpromotion', 'type': 'confounder', 'weight': None},
        {'from': 'cluster', 'to': 'is_holiday', 'type': 'confounder', 'weight': None},
        {'from': 'cluster', 'to': 'onpromotion', 'type': 'confounder', 'weight': None},
        {'from': 'oil_price', 'to': 'is_holiday', 'type': 'confounder', 'weight': None},
        {'from': 'oil_price', 'to': 'onpromotion', 'type': 'confounder', 'weight': None},
        {'from': 'month', 'to': 'is_holiday', 'type': 'confounder', 'weight': None},
        {'from': 'month', 'to': 'onpromotion', 'type': 'confounder', 'weight': None}
    ]
}

with open('output_causal_graph.json', 'w') as f:
    json.dump(causal_graph, f, indent=2)
print("✓ Exported: output_causal_graph.json")



PHASE 11: EXPORTING DATA FOR UI COMPONENTS

✓ Exported: output_dimension_view.csv
✓ Exported: output_causal_estimates.csv
✓ Exported: output_balance_table.csv
✓ Exported: output_regression_table.csv
✓ Exported: output_mediation_analysis.csv
✓ Exported: output_heterogeneous_effects.csv
✓ Exported: output_graph_data.csv


PHASE 12: CAUSAL GRAPH SPECIFICATION (DAG)

✓ Exported: output_causal_graph.json


In [2]:
# Set style
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'sans-serif'

# 1. CAUSAL ESTIMATES COMPARISON
try:
    estimates_df = pd.read_csv('output_causal_estimates.csv')
    # Filter only Naive and Adjusted
    comp_df = estimates_df[estimates_df['Method'].isin(['Naive OLS', 'Adjusted OLS'])].copy()
    
    plt.figure(figsize=(10, 6))
    colors = ['#ff9999', '#66b3ff']
    sns.barplot(x='Method', y='Estimate', data=comp_df, palette=colors)
    plt.title('Perbandingan Estimasi Naif vs Kausal (ATE)', fontsize=14, fontweight='bold')
    plt.ylabel('Average Treatment Effect (Poin Persentase)')
    plt.xlabel('')
    for i, v in enumerate(comp_df['Estimate']):
        plt.text(i, v + 0.01, f"{v:.4f}", ha='center', fontweight='bold')
    plt.savefig('viz_causal_estimates.png', dpi=300, bbox_inches='tight')
    plt.close()
except Exception as e:
    print(f"Error Estimates: {e}")

# 2. MEDIATION ANALYSIS
try:
    mediation_df = pd.read_csv('output_mediation_analysis.csv')
    # Use Percentage for Direct and Indirect
    paths = mediation_df[mediation_df['Path'].isin(["Direct Effect (c')", "Indirect Effect (a×b)"])]
    
    plt.figure(figsize=(8, 8))
    plt.pie(paths['Percentage'], labels=['Direct Effect (Proaktif)', 'Indirect Effect (Reaktif)'], 
            autopct='%1.1f%%', startangle=140, colors=['#99ff99', '#ffcc99'],
            explode=(0.1, 0), shadow=True)
    plt.title('Proporsi Mekanisme Keputusan Promosi', fontsize=14, fontweight='bold')
    plt.savefig('viz_mediation.png', dpi=300, bbox_inches='tight')
    plt.close()
except Exception as e:
    print(f"Error Mediation: {e}")

# 3. HETEROGENEITY - STORE TYPE
try:
    dim_df = pd.read_csv('output_dimension_view.csv')
    store_df = dim_df[dim_df['Dimension'] == 'Store Type'].sort_values('Difference', ascending=False)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(x='Category', y='Difference', data=store_df, palette='viridis')
    plt.title('Dampak Hari Libur Berdasarkan Tipe Toko', fontsize=14, fontweight='bold')
    plt.ylabel('Kenaikan Promosi (Diff)')
    plt.xlabel('Tipe Toko')
    plt.savefig('viz_dimension_store.png', dpi=300, bbox_inches='tight')
    plt.close()
except Exception as e:
    print(f"Error Store Type: {e}")

# 4. HETEROGENEITY - TOP 10 CLUSTERS
try:
    cluster_df = dim_df[dim_df['Dimension'] == 'Cluster'].sort_values('Difference', ascending=False).head(10)
    
    plt.figure(figsize=(12, 6))
    sns.barplot(x='Category', y='Difference', data=cluster_df, palette='rocket')
    plt.title('Top 10 Klaster Toko Paling Responsif Terhadap Hari Libur', fontsize=14, fontweight='bold')
    plt.ylabel('Kenaikan Promosi (Diff)')
    plt.xlabel('Klaster')
    plt.xticks(rotation=45)
    plt.savefig('viz_dimension_cluster.png', dpi=300, bbox_inches='tight')
    plt.close()
except Exception as e:
    print(f"Error Cluster: {e}")

# 5. POLICY SIMULATION (COUNTERFACTUAL)
try:
    # Based on data provided in user prompt: Current: 2.6028, All Holidays: 2.7705, No Holidays: 2.5745
    scenarios = ['Tanpa Hari Libur', 'Kondisi Saat Ini', 'Semua Hari Libur']
    values = [2.5745, 2.6028, 2.7705]
    
    plt.figure(figsize=(10, 6))
    plt.plot(scenarios, values, marker='o', linestyle='-', color='teal', linewidth=3, markersize=10)
    plt.fill_between(scenarios, values, color='teal', alpha=0.1)
    plt.title('Simulasi Counterfactual: Skenario Kebijakan Promosi', fontsize=14, fontweight='bold')
    plt.ylabel('Rata-rata Partisipasi Promosi')
    plt.ylim(2.5, 2.85)
    for i, v in enumerate(values):
        plt.text(i, v + 0.01, f"{v:.4f}", ha='center', fontweight='bold')
    plt.savefig('viz_counterfactual.png', dpi=300, bbox_inches='tight')
    plt.close()
except Exception as e:
    print(f"Error Counterfactual: {e}")

# 6. CAUSAL DAG (User provided code)
try:
    with open('output_causal_graph.json', 'r') as f:
        graph_data = json.load(f)

    G = nx.DiGraph()
    color_map_nodes = {
        'treatment': '#ff9999',  
        'outcome': '#99ff99',    
        'confounder': '#9999ff', 
        'mediator': '#ffff99'    
    }

    node_colors = []
    labels = {}
    for node in graph_data['nodes']:
        G.add_node(node['id'])
        labels[node['id']] = node['label']
        node_colors.append(color_map_nodes.get(node['type'], '#cccccc'))

    edge_labels = {}
    for edge in graph_data['edges']:
        u, v = edge['from'], edge['to']
        G.add_edge(u, v)
        if edge.get('weight') is not None:
            edge_labels[(u, v)] = f"{edge['weight']:.4f}"

    pos = {
        'store_type': (-1.5, 2),
        'cluster': (-0.5, 2),
        'oil_price': (0.5, 2),
        'month': (1.5, 2),
        'is_holiday': (-1, 0),
        'transactions': (1, 0),
        'onpromotion': (0, -2)
    }

    plt.figure(figsize=(14, 10))
    nx.draw_networkx_nodes(G, pos, node_size=5000, node_color=node_colors, 
                           alpha=0.9, edgecolors='black', linewidths=1.5)
    nx.draw_networkx_edges(G, pos, arrowstyle='->', arrowsize=30, 
                           edge_color='gray', width=2, 
                           connectionstyle="arc3,rad=0.1")
    nx.draw_networkx_labels(G, pos, labels, font_size=10, font_weight='bold')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, 
                                 font_size=11, label_pos=0.3, font_color='red')

    plt.title("Causal DAG: Seller Decision on Promotion Participation", 
              fontsize=16, fontweight='bold', pad=20)
    plt.axis('off')
    plt.savefig('causal_dag_visualization.png', dpi=300, bbox_inches='tight')
    plt.close()
except Exception as e:
    print(f"Error DAG: {e}")